# NASA C-MAPSS Turbofan Engine Degradation Dataset

**Source:** Saxena, A. and Goebel, K. (2008). *Turbofan Engine Degradation Simulation Data Set.*  
NASA Prognostics Data Repository, NASA Ames Research Center, Moffett Field, CA.

**C-MAPSS** = Commercial Modular Aero-Propulsion System Simulation

---

## What is this dataset?

This dataset simulates the **run-to-failure degradation** of turbofan jet engines. Each engine starts healthy (with some initial manufacturing variation) and degrades over time until it fails. The simulation captures 21 sensor readings per flight cycle.

**The task:** Given sensor readings up to some point in an engine's life, predict how many cycles remain before failure — this is called the **Remaining Useful Life (RUL)**.

This is a classic **Predictive Health Monitoring (PHM)** regression problem. It was originally designed for the PHM 2008 Data Challenge.

## Dataset Structure

There are **4 sub-datasets** (FD001 – FD004), each representing different operating conditions and fault types:

| Dataset | Engines (test) | Operating Conditions | Fault Modes | Rows (test) |
|---------|---------------|----------------------|-------------|-------------|
| FD001   | 100           | 1 (sea level)        | HPC degradation only | ~13,096 |
| FD002   | 259           | 6 different          | HPC degradation only | ~33,991 |
| FD003   | 100           | 1 (sea level)        | HPC + Fan degradation | ~16,596 |
| FD004   | 248           | 6 different          | HPC + Fan degradation | ~41,214 |

**Difficulty increases** from FD001 → FD004:  
- More operating conditions = sensors shift between flights for non-degradation reasons  
- More fault modes = multiple failure paths to learn

**FD001 is the standard benchmark** used in most research papers.

## File Format

Each sub-dataset comes with two files:

| File | Description |
|------|-------------|
| `test_FDxxx.txt` | Sensor readings for each engine, truncated at a random cycle |
| `RUL_FDxxx.txt`  | One integer per engine — cycles remaining at truncation |

Files are **space-separated**, no header row, **26 columns** per row in the test files.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

# Column names for the dataset
col_names = (
    ['unit_id', 'cycle'] +
    [f'op_setting_{i}' for i in range(1, 4)] +
    [f'sensor_{i}' for i in range(1, 22)]
)

print(f'Total columns: {len(col_names)}')
print(f'Columns: {col_names}')

## Loading the Data

Update the `DATA_PATH` variable to point to the folder containing your downloaded C-MAPSS files.

In [ ]:
DATA_PATH = './data/'  # <-- update this path

def load_dataset(fd_id):
    """Load test data and RUL labels for a given sub-dataset (1-4)."""
    test = pd.read_csv(
        f'{DATA_PATH}test_FD00{fd_id}.txt',
        sep=' ', header=None, names=col_names,
        index_col=False
    ).dropna(axis=1, how='all')  # drop trailing empty columns
    
    rul = pd.read_csv(
        f'{DATA_PATH}RUL_FD00{fd_id}.txt',
        sep=' ', header=None, names=['rul']
    ).dropna(axis=1, how='all')
    
    return test, rul

test_fd1, rul_fd1 = load_dataset(1)

print(f'Test shape: {test_fd1.shape}')
print(f'RUL shape:  {rul_fd1.shape}')
print(f'\nNumber of unique engines: {test_fd1["unit_id"].nunique()}')
print(f'Cycles per engine (min/max): {test_fd1.groupby("unit_id")["cycle"].max().min()} / {test_fd1.groupby("unit_id")["cycle"].max().max()}')

In [ ]:
# Preview the first few rows
test_fd1.head(5)

## The 26 Columns Explained

| Column | Name | Description |
|--------|------|-------------|
| 1  | `unit_id` | Engine unit number (1 to N) |
| 2  | `cycle` | Flight cycle number (time step) |
| 3  | `op_setting_1` | Altitude (ft) |
| 4  | `op_setting_2` | Throttle resolver angle (°) |
| 5  | `op_setting_3` | Fan speed command (rpm) |
| 6  | `sensor_1` | T2 — Total temperature at fan inlet (°R) |
| 7  | `sensor_2` | T24 — Total temperature at LPC outlet (°R) |
| 8  | `sensor_3` | T30 — Total temperature at HPC outlet (°R) |
| 9  | `sensor_4` | T50 — Total temperature at LPT outlet (°R) |
| 10 | `sensor_5` | P2 — Pressure at fan inlet (psia) |
| 11 | `sensor_6` | P15 — Total pressure in bypass-duct (psia) |
| 12 | `sensor_7` | P30 — Total pressure at HPC outlet (psia) |
| 13 | `sensor_8` | Nf — Physical fan speed (rpm) |
| 14 | `sensor_9` | Nc — Physical core speed (rpm) |
| 15 | `sensor_10` | epr — Engine pressure ratio |
| 16 | `sensor_11` | Ps30 — Static pressure at HPC outlet (psia) |
| 17 | `sensor_12` | phi — Fuel flow ratio (pps/psi) |
| 18 | `sensor_13` | NRf — Corrected fan speed (rpm) |
| 19 | `sensor_14` | NRc — Corrected core speed (rpm) |
| 20 | `sensor_15` | BPR — Bypass ratio |
| 21 | `sensor_16` | farB — Burner fuel-air ratio |
| 22 | `sensor_17` | htBleed — Bleed enthalpy |
| 23 | `sensor_18` | Nf_dmd — Demanded fan speed (rpm) |
| 24 | `sensor_19` | PCNfR_dmd — Demanded corrected fan speed |
| 25 | `sensor_20` | W31 — HPT coolant bleed (lbm/s) |
| 26 | `sensor_21` | W32 — LPT coolant bleed (lbm/s) |

> **HPC** = High Pressure Compressor, **LPC** = Low Pressure Compressor  
> **HPT** = High Pressure Turbine, **LPT** = Low Pressure Turbine

## Identifying Flat (Uninformative) Sensors

Not all 21 sensors are useful. Several have near-zero standard deviation in FD001, meaning they carry no degradation signal. **Dropping these before training is standard practice.**

In [ ]:
sensor_cols = [f'sensor_{i}' for i in range(1, 22)]
op_cols = [f'op_setting_{i}' for i in range(1, 4)]

# Compute std for all feature columns
feature_cols = op_cols + sensor_cols
stds = test_fd1[feature_cols].std().sort_values()

# Classify as flat if std < 0.01
flat_cols = stds[stds < 0.01].index.tolist()
useful_cols = stds[stds >= 0.01].index.tolist()

print(f'Flat / near-constant features ({len(flat_cols)}): {flat_cols}')
print(f'\nUseful features ({len(useful_cols)}): {useful_cols}')

# Visualise
fig, ax = plt.subplots(figsize=(12, 4))
colors = ['#F09595' if s < 0.01 else '#85B7EB' for s in stds.values]
ax.bar(range(len(stds)), stds.values, color=colors, edgecolor='none')
ax.set_xticks(range(len(stds)))
ax.set_xticklabels(stds.index, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Standard deviation')
ax.set_title('Feature variability in FD001 test set')
ax.axhline(0.01, color='gray', linestyle='--', linewidth=0.8, label='flat threshold (0.01)')
ax.legend(fontsize=9)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#F09595', label=f'Flat — drop ({len(flat_cols)})'),
    Patch(facecolor='#85B7EB', label=f'Useful — keep ({len(useful_cols)})')
]
ax.legend(handles=legend_elements, fontsize=9)
plt.tight_layout()
plt.show()

## RUL Label Distribution

Each line in `RUL_FDxxx.txt` is one integer — the cycles remaining for that engine at the point the test sequence was cut off.

The key observation: **some engines are cut off very close to failure (RUL = 6–7)**, making those predictions the hardest to get right.

In [ ]:
# Load all four RUL files and compare
rul_data = {}
for fd_id in range(1, 5):
    try:
        _, rul = load_dataset(fd_id)
        rul_data[f'FD00{fd_id}'] = rul['rul'].values
    except FileNotFoundError:
        print(f'FD00{fd_id} not found — skipping')

# Summary statistics
stats = pd.DataFrame({
    ds: {'count': len(v), 'min': v.min(), 'mean': round(v.mean(), 1), 'max': v.max()}
    for ds, v in rul_data.items()
}).T

print('RUL statistics across datasets:')
print(stats.to_string())

# Plot distributions
fig, axes = plt.subplots(1, len(rul_data), figsize=(14, 4), sharey=False)
colors = ['#7F77DD', '#1D9E75', '#BA7517', '#D85A30']

for ax, (name, vals), color in zip(axes, rul_data.items(), colors):
    ax.hist(vals, bins=20, color=color, alpha=0.85, edgecolor='white', linewidth=0.5)
    ax.axvline(vals.mean(), color='black', linestyle='--', linewidth=1, label=f'mean={vals.mean():.0f}')
    ax.set_title(name, fontsize=11)
    ax.set_xlabel('RUL (cycles)')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('RUL distribution per sub-dataset', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## Engine Degradation Over Time

Let's visualize how a few key sensors degrade as an engine approaches failure.

To do this we need the **training data** (which runs engines all the way to failure). The test data is truncated, so we construct a proxy RUL for the test set by assuming each engine's last observed cycle corresponds to its ground-truth RUL from the RUL file.

In [ ]:
# Attach RUL to each row in the test set
# The RUL file gives cycles remaining at the LAST row of each engine
# So: RUL at row t = (max_cycle_for_engine - cycle_at_row_t) + ground_truth_RUL

def add_rul(test_df, rul_df):
    max_cycles = test_df.groupby('unit_id')['cycle'].max().reset_index()
    max_cycles.columns = ['unit_id', 'max_cycle']
    
    rul_df = rul_df.copy()
    rul_df['unit_id'] = range(1, len(rul_df) + 1)
    
    df = test_df.merge(max_cycles, on='unit_id').merge(rul_df, on='unit_id')
    df['rul_true'] = (df['max_cycle'] - df['cycle']) + df['rul']
    return df

fd1 = add_rul(test_fd1, rul_fd1)

# Plot 3 informative sensors for 5 random engines
sensors_to_plot = ['sensor_2', 'sensor_3', 'sensor_4']  # T24, T30, T50
sensor_labels  = ['T24 — LPC outlet temp', 'T30 — HPC outlet temp', 'T50 — LPT outlet temp']

sample_engines = fd1['unit_id'].unique()[:5]
colors_eng = ['#7F77DD','#1D9E75','#BA7517','#D85A30','#D4537E']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, sensor, label in zip(axes, sensors_to_plot, sensor_labels):
    for eng, color in zip(sample_engines, colors_eng):
        eng_data = fd1[fd1['unit_id'] == eng].sort_values('cycle')
        ax.plot(eng_data['rul_true'].values[::-1],  # x = RUL counting down
                eng_data[sensor].values,
                color=color, alpha=0.7, linewidth=1)
    ax.invert_xaxis()  # RUL decreases left → right toward failure
    ax.set_xlabel('RUL (cycles remaining)')
    ax.set_ylabel(sensor)
    ax.set_title(label, fontsize=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Sensor readings as engines approach failure (FD001, 5 engines)', fontsize=11, y=1.02)
plt.tight_layout()
plt.show()
print('Note: x-axis goes right → left (closer to 0 = closer to failure)')

## The Piece-Wise Linear RUL Target

A key preprocessing decision for this dataset: **cap the RUL target at 125 cycles**.

**Why?** In early life, a healthy engine's sensor readings look nearly identical whether it has 200 or 400 cycles left. The model can't learn anything useful from very-early-life data. Capping at 125 tells the model: "anything above 125 cycles is just 'healthy' — only start learning the degradation curve after that."

This is called the **piece-wise linear (PWL) target** and is used in nearly all research on this dataset.

In [ ]:
RUL_CAP = 125

fd1['rul_capped'] = fd1['rul_true'].clip(upper=RUL_CAP)

# Show one engine's raw vs capped RUL
eng1 = fd1[fd1['unit_id'] == 1].sort_values('cycle')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(eng1['cycle'], eng1['rul_true'],  label='Raw RUL',    color='#B4B2A9', linewidth=1.5, linestyle='--')
ax.plot(eng1['cycle'], eng1['rul_capped'], label='Capped RUL (PWL target)', color='#7F77DD', linewidth=2)
ax.axhline(RUL_CAP, color='#D85A30', linestyle=':', linewidth=1, label=f'Cap = {RUL_CAP}')
ax.set_xlabel('Cycle')
ax.set_ylabel('RUL')
ax.set_title('Raw vs piece-wise linear RUL target (Engine 1, FD001)')
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## Sensor Correlation with RUL

Which sensors are most correlated with degradation? We compute Pearson correlation between each sensor and the capped RUL. Strong negative correlation = sensor increases as engine degrades (or vice versa).

In [ ]:
# Only use the useful sensors
useful_sensors = [c for c in useful_cols if c.startswith('sensor')]

corr = fd1[useful_sensors + ['rul_capped']].corr()['rul_capped'].drop('rul_capped').sort_values()

fig, ax = plt.subplots(figsize=(10, 4))
bar_colors = ['#F09595' if v < 0 else '#9FE1CB' for v in corr.values]
ax.barh(range(len(corr)), corr.values, color=bar_colors, edgecolor='none')
ax.set_yticks(range(len(corr)))
ax.set_yticklabels(corr.index, fontsize=9)
ax.axvline(0, color='var(--b)', linewidth=0.8)
ax.set_xlabel('Pearson correlation with capped RUL')
ax.set_title('Sensor correlation with RUL (FD001)')

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor='#F09595', label='Negative: sensor rises as engine degrades'),
    Patch(facecolor='#9FE1CB', label='Positive: sensor drops as engine degrades')
], fontsize=9)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## Standard Preprocessing Pipeline

Here is the standard preprocessing recipe used before feeding this data into an LSTM or other model:

1. **Drop flat sensors** — remove near-constant columns (std < 0.01)
2. **Cap RUL at 125** — piece-wise linear target
3. **Normalize sensors** — min-max scale each sensor to [0, 1] using training set statistics
4. **Create sliding windows** — group each engine into overlapping windows of 30 cycles
5. **Label each window** — target RUL = RUL at the last timestep of each window

In [ ]:
from sklearn.preprocessing import MinMaxScaler

WINDOW_SIZE = 30
RUL_CAP = 125

def preprocess(test_df, rul_df, window_size=30, rul_cap=125):
    """Full preprocessing pipeline: drop flat cols, cap RUL, normalize, window."""
    
    # 1. Attach RUL to rows
    df = add_rul(test_df, rul_df)
    df['rul_capped'] = df['rul_true'].clip(upper=rul_cap)
    
    # 2. Identify and drop flat columns
    feature_cols = [f'op_setting_{i}' for i in range(1, 4)] + [f'sensor_{i}' for i in range(1, 22)]
    stds = df[feature_cols].std()
    keep_cols = stds[stds >= 0.01].index.tolist()
    
    # 3. Normalize
    scaler = MinMaxScaler()
    df[keep_cols] = scaler.fit_transform(df[keep_cols])
    
    # 4. Create sliding windows
    X, y = [], []
    for engine_id in df['unit_id'].unique():
        engine_data = df[df['unit_id'] == engine_id].sort_values('cycle')
        sensor_vals = engine_data[keep_cols].values
        rul_vals = engine_data['rul_capped'].values
        
        # Only create windows if engine has enough cycles
        if len(sensor_vals) >= window_size:
            for i in range(len(sensor_vals) - window_size + 1):
                X.append(sensor_vals[i : i + window_size])
                y.append(rul_vals[i + window_size - 1])
    
    return np.array(X), np.array(y), keep_cols, scaler

X, y, kept_features, scaler = preprocess(test_fd1, rul_fd1)

print(f'Input shape (windows, timesteps, features): {X.shape}')
print(f'Target shape: {y.shape}')
print(f'Features kept: {len(kept_features)}')
print(f'RUL range in targets: [{y.min():.0f}, {y.max():.0f}]')

## Why LSTM?

A single row of sensor data tells you the current state of the engine. But **degradation is a trajectory** — the rate of change matters, not just the current value. An engine with rapidly rising temperatures is more concerning than one with the same current temperature but a stable history.

**LSTM (Long Short-Term Memory)** is ideal here because:
- It processes sequences of arbitrary length
- It learns what to remember and forget across timesteps (via gates)
- It can detect subtle trends in noisy sensor data over a window of 30 cycles

The input to your LSTM model has shape: `(batch_size, 30, n_features)` — 30 timesteps, each with `n_features` sensor readings.

In [ ]:
# Visualize what one input window looks like
sample_window = X[100]  # pick an arbitrary window
sample_rul = y[100]

fig, ax = plt.subplots(figsize=(12, 4))
for i in range(sample_window.shape[1]):
    ax.plot(range(WINDOW_SIZE), sample_window[:, i], alpha=0.4, linewidth=0.8)

ax.set_xlabel('Timestep within window (0 = oldest, 29 = most recent)')
ax.set_ylabel('Normalized sensor value [0, 1]')
ax.set_title(f'One input window — {sample_window.shape[1]} sensors over {WINDOW_SIZE} cycles | Target RUL = {sample_rul:.0f}')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print(f'Window shape: {sample_window.shape}  →  fed as one sample to the LSTM')

## Summary

| Aspect | Detail |
|--------|--------|
| **Task** | Regression — predict RUL (cycles to failure) |
| **Input** | Sequence of 30 cycles × 14 useful sensor features |
| **Output** | Single integer — RUL at the last timestep |
| **Evaluation metrics** | RMSE, MAE, and PHM score (asymmetric — penalises late predictions more) |
| **Key benchmark** | FD001 (single condition, single fault) |
| **RUL cap** | 125 cycles (piece-wise linear target) |
| **Flat sensors to drop** | 10 near-constant columns in FD001 |
| **Best reported models** | LSTM, Bi-LSTM, Transformer, CNN-LSTM hybrids |

### PHM Asymmetric Scoring Function

The original challenge used a scoring function that penalises **late predictions more than early ones** — because predicting an engine will last longer than it actually does is dangerous in practice:

$$s = \sum_{i=1}^{N} \begin{cases} e^{-d/13} - 1 & \text{if } d < 0 \\ e^{d/10} - 1 & \text{if } d \geq 0 \end{cases}$$

where $d = \hat{RUL} - RUL_{true}$ (positive = predicted too high = late = dangerous).

In [ ]:
def phm_score(y_true, y_pred):
    """PHM 2008 asymmetric scoring function."""
    d = y_pred - y_true
    scores = np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)
    return np.sum(scores)

# Visualize the asymmetry
d_range = np.linspace(-50, 50, 300)
score_curve = np.where(d_range < 0, np.exp(-d_range / 13) - 1, np.exp(d_range / 10) - 1)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(d_range[d_range < 0],  score_curve[d_range < 0],  color='#9FE1CB', linewidth=2, label='Early prediction (safe)')
ax.plot(d_range[d_range >= 0], score_curve[d_range >= 0], color='#F09595', linewidth=2, label='Late prediction (dangerous)')
ax.axvline(0, color='gray', linestyle='--', linewidth=0.8)
ax.set_xlabel('d = predicted RUL − true RUL')
ax.set_ylabel('Penalty score')
ax.set_title('PHM asymmetric scoring function — late predictions penalised more')
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()